#**Reto computacional I:** El plano invariable de Laplace del Sistema Solar

<div align="left">


## *Mecánica Celeste (2024-2)*
### Juliana Ruiz Montoya
### 1007435437

El reto consiste en calcular la inclinación del plano invariable de Laplace del Sistema Solar con respecto a la órbita de los planetas del Sistema Solar. Para esto:

* Escoja como fecha, el día de su cumpleaños en 2024 y como hora las 0:00 de UT.

* Use astroquery (rutina consulta_horizons de pymcel) para obtener las posiciones y velocidades del Sol y los 8 planetas mayores del Sistema solar.

* Use SPICE para conseguir el valor de los μ de los cuerpos. Debe tener en cuenta que SPICE da el valor de estas cantidades en km³/s² pero para el cálculo usted necesita esa cantidad en m³/s². Haga el cambio de unidades.

* Calcule con esta información el valor de las componentes x, y, z del momentum angular total del Sistema Solar.

* Calcule el momentum angular de cada planeta y encuentre el ángulo que forma cada uno con respecto al momentum angular total del sistema. Ese ángulo define la orientación del plano de Laplace. Ayuda: el ángulo se puede calcular usando la definición de producto punto.

Opcional: calcule la latitud y longitud eclíptica del polo del plano de Laplace, es decir, la ubicación del vector del momentum angular total sobre la esfera celeste.

In [1]:
!pip install -Uq pymcel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 18.4 MB/s eta 0:00:00


In [2]:
!pip install -Uq rebound celluloid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.3/785.3 kB 12.8 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import pymcel as pc
import rebound as rb
import matplotlib.pyplot as plt
from IPython.display import HTML
from celluloid import Camera

Paquete pymcel cargado. Versión: 0.6.10


In [4]:
pc.descarga_kernels()

In [5]:
import spiceypy as spy

In [6]:
spy.furnsh([
    'pymcel/data/gm_de431.tpc',
    'pymcel/data/de430.bsp',
    'pymcel/data/latest_leapseconds.tls'
])

Se crea el diccionario *planet_datos* para almacenar los datos de los planetas y se crea otro diccionario *mu_planeta* para almacenar los μ de cada planeta, siendo $μ = GM$, luego se consulta con `pc.consulta_horizonts` los vectores de estado (vector posición $\vec{r}$ y el vector velocidad $\vec{v}$) de cada uno para almacenarlos en el diccionario *planet_datos* para poder consultar cada vector correspondiente al id de su planeta; y por último con `SPICE` se obtienen los μ y se almacenan en el diccionario *mu_planetas*.

In [7]:
planet_datos = {}
mu_planeta = {}

for id in ['Sun',
           'Mercury Barycenter',
           'Venus Barycenter',
           'Earth-Moon Barycenter',
           'Mars Barycenter',
           'Jupiter Barycenter',
           'Saturn Barycenter',
           'Uranus Barycenter',
           'Neptune Barycenter']:
  tabla, df, datos = pc.consulta_horizons(
    id=id,
    location='@0',
    epochs='2024-10-31 00:00:00',
    datos='vectors'
  )
  mu_cuerpo = spy.bodvrd(id,'GM',1)[1][0]

  mu_planeta[id] = mu_cuerpo * 1000 # cambio de km^3 / s^2 a m^3 / s^2
  planet_datos[id] = datos

In [8]:
planet_datos # vector posicion y velocidad de cada id [m y m/s]

{'Sun': array([-9.22313414e+08, -6.97157771e+08,  2.78942338e+07,  1.18320298e+01,
        -7.57637262e+00, -1.81751999e-01]),
 'Mercury Barycenter': array([ 8.42339559e+09, -6.84413302e+10, -6.36545162e+09,  3.84973905e+04,
         9.14789154e+03, -2.78196530e+03]),
 'Venus Barycenter': array([ 7.25131980e+10, -8.09846933e+10, -5.31204829e+09,  2.56255515e+04,
         2.35006384e+04, -1.15526052e+03]),
 'Earth-Moon Barycenter': array([ 1.16566218e+11,  9.01576810e+10,  2.21059786e+07, -1.86960889e+04,
         2.34451512e+04, -1.39929922e+00]),
 'Mars Barycenter': array([ 4.41141486e+10,  2.25477313e+11,  3.66285565e+09, -2.28356251e+04,
         6.78339959e+03,  7.02480635e+02]),
 'Jupiter Barycenter': array([ 2.25609572e+11,  7.21961463e+11, -8.04224247e+09, -1.26188359e+04,
         4.51964012e+03,  2.63598069e+02]),
 'Saturn Barycenter': array([ 1.40695646e+12, -3.15292334e+11, -5.05358341e+10,  1.57505794e+03,
         9.40596140e+03, -2.26272274e+02]),
 'Uranus Barycenter': ar

In [9]:
mu_planeta # mu de cada id [m^3 / s^2]

{'Sun': 132712440041939.3,
 'Mercury Barycenter': 22031780.000000022,
 'Venus Barycenter': 324858592.0,
 'Earth-Moon Barycenter': 403503235.5022598,
 'Mars Barycenter': 42828375.214000024,
 'Jupiter Barycenter': 126712764800.0002,
 'Saturn Barycenter': 37940585200.0,
 'Uranus Barycenter': 5794548600.000008,
 'Neptune Barycenter': 6836527100.580023}

Se cambian los diccionarios que contienen los vectores de estado a arrays de `numpy`, luego se almacenan las posiciones de cada planeta en la variable Rs y las velocidades de cada planeta en Vs

In [10]:
Rs = np.array([planet_datos[id][0:3] for id in planet_datos])
Vs = np.array([planet_datos[id][3:6] for id in planet_datos])

In [11]:
Rs

array([[-9.22313414e+08, -6.97157771e+08,  2.78942338e+07],
       [ 8.42339559e+09, -6.84413302e+10, -6.36545162e+09],
       [ 7.25131980e+10, -8.09846933e+10, -5.31204829e+09],
       [ 1.16566218e+11,  9.01576810e+10,  2.21059786e+07],
       [ 4.41141486e+10,  2.25477313e+11,  3.66285565e+09],
       [ 2.25609572e+11,  7.21961463e+11, -8.04224247e+09],
       [ 1.40695646e+12, -3.15292334e+11, -5.05358341e+10],
       [ 1.69039320e+12,  2.38777086e+12, -1.30312001e+10],
       [ 4.46859025e+12, -1.24887267e+11, -1.00411389e+11]])

In [12]:
Vs

array([[ 1.18320298e+01, -7.57637262e+00, -1.81751999e-01],
       [ 3.84973905e+04,  9.14789154e+03, -2.78196530e+03],
       [ 2.56255515e+04,  2.35006384e+04, -1.15526052e+03],
       [-1.86960889e+04,  2.34451512e+04, -1.39929922e+00],
       [-2.28356251e+04,  6.78339959e+03,  7.02480635e+02],
       [-1.26188359e+04,  4.51964012e+03,  2.63598069e+02],
       [ 1.57505794e+03,  9.40596140e+03, -2.26272274e+02],
       [-5.60825413e+03,  3.61743352e+03,  8.60877708e+01],
       [ 1.15966781e+02,  5.46538734e+03, -1.15222225e+02]])

Se hace una lista por comprensión para almacenar las masas de cada planeta, lo que corresponde a dividir cada μ por la constante gravitacional $G$

In [13]:
masas_planetas = np.array([mu_planeta[id] for id in planet_datos])
masas_planetas = masas_planetas / pc.constantes.G
masas_planetas

array([1.98840987e+24, 3.30098737e+17, 4.86730581e+18, 6.04562629e+18,
       6.41690892e+17, 1.89851767e+21, 5.68457894e+20, 8.68188214e+19,
       1.02430623e+20])

Se almacena en la variable *L_planetas* el momento angular de cada cuerpo, el cual se define así:

$$\vec{L} = ∑_i m_i (\vec{r}_i × \vec{v}_i)$$

donde $m$ son las masas de cada cuerpo, $\vec{r}$ son los vectores posición y $\vec{v}$ son los vectores velocidad.

In [14]:
L_planetas = np.cross(Rs, Vs) * masas_planetas[:,None]
L_planetas

array([[ 6.72175848e+32,  3.22943810e+32,  3.02965694e+34],
       [ 8.20730652e+31, -7.31563773e+31,  8.95184507e+32],
       [ 1.06299499e+33, -2.54815782e+32,  1.83954171e+34],
       [-3.89601670e+30, -1.51252093e+30,  2.67126519e+34],
       [ 8.56957863e+31, -7.35589094e+31,  3.49603420e+33],
       [ 4.30309830e+35,  7.97633546e+34,  1.92319599e+37],
       [ 3.10764572e+35,  1.35724080e+35,  7.80514395e+36],
       [ 2.19388879e+34, -6.28914685e+33,  1.69349755e+36],
       [ 5.76865635e+34,  5.15468311e+34,  2.50310323e+36]])

Se almacena el momento angular total del sistema solar en la variable *L_total* sumando componente a componente del momento angular de cada cuerpo.

In [15]:
L_total = [sum(L_planetas[:, i]) for i in range(3)]
L_total = np.array(L_total)
print("Momentum angular total del Sistema Solar:", L_total)

Momentum angular total del Sistema Solar: [8.22598897e+35 2.60665019e+35 3.13135005e+37]


In [16]:
print("Momentum angular total del Sistema Solar en la componente x:",L_total[0])
print("Momentum angular total del Sistema Solar en la componente y:",L_total[1])
print("Momentum angular total del Sistema Solar en la componente z:",L_total[2])

Momentum angular total del Sistema Solar en la componente x: 8.2259889717867e+35
Momentum angular total del Sistema Solar en la componente y: 2.6066501929829528e+35
Momentum angular total del Sistema Solar en la componente z: 3.1313500507259374e+37


Ahora, para hallar el ángulo que forma cada momentum angular de los cuerpos con respecto al momentum angular total del sistema se usa la definición de producto punto:

 $$ \cos \theta = \frac{\vec{L_{cuerpo}} \cdot \vec{L_{total}}}{\|\vec{L_{cuerpo}}\| \|\vec{L_{total}}\|} $$




In [17]:
import math

In [18]:
l_norma_planeta = []

for planeta in L_planetas:
  norma = np.linalg.norm(planeta)

  l_norma_planeta.append(norma)

In [19]:
l_norma_planeta

[3.0305745783424192e+34,
 9.019108303720579e+32,
 1.8427866374748684e+34,
 2.6712652228359845e+34,
 3.4978578869128854e+33,
 1.923693871817711e+37,
 7.812507131244726e+36,
 1.693651330170875e+36,
 2.5042984211503122e+36]

In [20]:
angulos = ([np.arccos(np.dot(L_planetas, L_total) / (np.array(l_norma_planeta) * np.linalg.norm(L_total)))])
print("Estos son los ángulos correspondientes a cada cuerpo:", angulos)

Estos son los ángulos correspondientes a cada cuerpo: [array([0.00470154, 0.11068964, 0.03847275, 0.02770632, 0.02940505,
       0.00570814, 0.01627847, 0.01794375, 0.01267955])]


In [21]:
a = angulos[0].min()
print("El ángulo mínimo es:", a)
print("El índice que corresponde a el ángulo mínimo es", np.argmin(angulos), "y este cuerpo es el sol")

El ángulo mínimo es: 0.0047015392915495825
El índice que corresponde a el ángulo mínimo es 0 y este cuerpo es el sol


In [22]:
angulos = np.array(angulos)
angulos_en_grados = np.degrees(angulos)
angulos_en_grados

array([[0.26937836, 6.3420492 , 2.2043264 , 1.58745525, 1.68478525,
        0.32705255, 0.93268739, 1.02810116, 0.72648471]])

De esta manera podemos concluir que el cuerpo con menor inclinación entre su vector de momento angular y el vector de momento angular total del sistema solar, es el sol, con una inclinación de 0.0047 rad, lo que son 0.269°.
Lo que confirma la definición de promedios pesados, ya que el cuerpo más masivo es el que indica la dirección del plano invariante.